-------->  LIBRARIES && Load Dataset

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv("Earthquakes_Dataset.csv")

print("First 5 Rows:")
print(data.head())

print("\nDataset Shape:", data.shape)

-------->     Check Missing Values

In [ ]:
print("Missing Values in Each Column:")
print(data.isnull().sum())

------->  Handle Missing Values

In [ ]:
# Fill numerical columns with median
num_cols = data.select_dtypes(include=np.number).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

# Fill categorical columns with mode
cat_cols = data.select_dtypes(include='object').columns

for col in cat_cols:
    data[col].fillna(data[col].mode()[0], inplace=True)

print("Missing values handled")

----->  Convert Date/Time Feature

In [ ]:
if 'time' in data.columns:

    data['time'] = pd.to_datetime(data['time'], errors='coerce')

    data['year'] = data['time'].dt.year
    data['month'] = data['time'].dt.month
    data['day'] = data['time'].dt.day

print(data[['year','month','day']].head())

------->   Feature Engineering

In [ ]:
# Create magnitude category
if 'mag' in data.columns:
    data['magnitude_level'] = pd.cut(
        data['mag'],
        bins=[0,4,6,10],
        labels=['Low','Moderate','High']
    )

print(data[['mag','magnitude_level']].head())

------>   Normalize Numerical Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

if 'mag' in data.columns and 'depth' in data.columns:
    data[['mag','depth']] = scaler.fit_transform(data[['mag','depth']])

print("Normalization completed")

------>   Remove Duplicate Data

In [ ]:
before = data.shape[0]

data = data.drop_duplicates()

after = data.shape[0]

print("Duplicates removed:", before-after)

-------->   Outlier Detection (IQR Method)

In [ ]:
if 'mag' in data.columns:
    Q1 = data['mag'].quantile(0.25)
    Q3 = data['mag'].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    data = data[(data['mag'] >= lower) & (data['mag'] <= upper)]

print("Outliers removed")

-------->  Save Preprocessed Dataset

In [ ]:
data.to_csv("preprocessed_earthquake_data.csv", index=False)

print("Preprocessed dataset saved successfully")

-------->  Final dataset shape

In [ ]:
print("Final Dataset Shape:", data.shape)
data.head()